In [1]:
from tokenizers import BertWordPieceTokenizer
from pathlib import Path
vocab_size = 30000
corpus_path = Path("../data/full_text.txt")
tokenizer_dir = "../data/models"
model_dir = "../data/models/bert-custom"
dataset_path = "../data/full_text.txt"
tokenizer = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=False,
    lowercase=True,
)

tokenizer.train(
    files=str(corpus_path),
    vocab_size=vocab_size,
    min_frequency=2,
    limit_alphabet=1000,
    wordpieces_prefix="##"
)

tokenizer.save_model(tokenizer_dir)

['../data/models/vocab.txt']

In [2]:
from transformers import BertConfig, BertForMaskedLM
config = BertConfig(
    vocab_size=vocab_size,
    max_position_embeddings=512,
    hidden_size=256,
    num_attention_heads=4,
    num_hidden_layers=2,
    type_vocab_size=2,
)
model = BertForMaskedLM(config)

/homes/mlugli/.conda/envs/dinoenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import Dataset
from transformers import BertTokenizerFast
def load_dataset():
    with open(dataset_path) as f:
        data = f.read().splitlines()
        lines = [line.strip() for line in data if line.strip()]
    return lines

def create_examples(lines):
    return [{"text": line} for line in lines if len(line.split()) > 10]

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

lines = load_dataset()
examples = create_examples(lines)
dataset = Dataset.from_list(examples)
tokenizer = BertTokenizerFast.from_pretrained(tokenizer_dir)
tokenized_dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 1107/1107 [00:00<00:00, 7580.29 examples/s]


In [4]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import os
os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir=model_dir,
    overwrite_output_dir=True,
    num_train_epochs=200,
    per_device_train_batch_size=32,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=10,
)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/homes/mlugli/.conda/envs/dinoenv/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
10,10.270000
20,10.071800
30,9.870200
40,9.695900
50,9.497600
60,9.308000
70,9.125600
80,8.942400
90,8.774100
100,8.597700


/homes/mlugli/.conda/envs/dinoenv/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


: 